In [ ]:
import logging
import nest_asyncio
import sys
from llama_index.core import Settings
from llama_index.llms.litellm import LiteLLM

from src.persona import qa_generator
from src.persona.qa_generator import QuestionType
from src.utils import load_util
from src.utils import rdf_util

logging.basicConfig(
    stream=sys.stdout, level=logging.WARN
)  # logging.INFO or even logging.DEBUG for more verbose output
# os.environ['LITELLM_LOG'] = 'INFO'

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

In [ ]:
# Output Path
data_path = './data'
qa_name = 'small_2'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config_2.yml') # persona_config_2.yml is the alternative config file which only allows food classes as ingredients in the persona

# Load key
api_key = load_util.load_file('config/api_key')

# 1. Setup

## 1.1 Load Preprocessed Data

In [ ]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path)
food_on = rdf_util.PreProcessedFoodOn(hummus_data).preprocess()

## 1.2 Init LLM

Alternatively a local LLM: [`llama-2-chat-13b-ggml`](https://huggingface.co/TheBloke/Llama-2-13B-chat-GGML) model.

Full setup instructions/details at [Llama CPP guide](https://gpt-index.readthedocs.io/en/stable/examples/llm/llama_2_llama_cpp.html).

In [ ]:
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1"
)

# Set Global Settings
Settings.llm = llm

# 2. Generate Questions

In [ ]:
# Generate QA set for recipes
recipe_question_answer_set = qa_generator.generate_question_answer_set(hummus_data, QuestionType.RECIPE, persona_config, nutrition_config, data_path=data_path, sub_dir_name=qa_name, llm=llm, size=10, batch_size=10, storage_frequency=10, food_on=food_on)

In [ ]:
# Remove duplicated questions (solely dependent on persona requirements) # Note: Not saved on disk
recipe_question_answer_set = qa_generator.remove_duplicates(recipe_question_answer_set)

In [ ]:
# Load, automatically adds full recipes instead of just the list of urls 
question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name)